<a href="https://colab.research.google.com/github/abhinavraghav008-cell/hcltech-project/blob/main/Abhinav_Raghav%2C2026B03110098%2CECE1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 12: Gym & Fitness Center Membership Analytics

**Industry:** Health & Wellness / Fitness  
**Goal:** Use member check-in logs, class attendance and membership tiers to improve facility scheduling and member retention.

**How to run (Google Colab):** run the cells from top to bottom. When the first code cell asks for a file, upload `12_gym_membership.csv`.

Programmes 1 to 15 are the required tasks. Programmes 16 and 17 are extra. Programme 18 saves the cleaned and randomized data and a results workbook.

In [ ]:
import numpy as np
import pandas as pd

# ---- Load the data file ----
# In Google Colab: a file picker appears. Choose 12_gym_membership.csv.
# Outside Colab: the file must be in the same folder as this notebook.
try:
    from google.colab import files
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
    print("Using uploaded file:", FILE_PATH)
except ImportError:
    FILE_PATH = "12_gym_membership.csv"
    print("Not running in Colab. Using local file:", FILE_PATH)


Saving 12_gym_membership.csv to 12_gym_membership.csv
Using uploaded file: 12_gym_membership.csv


## PROGRAMME 1: Load and inspect the membership dataset

In [ ]:
print("=" * 70)
print("PROGRAMME 1: Load and inspect the membership dataset")
print("=" * 70)

if FILE_PATH.endswith(".xlsx"):
    df = pd.read_excel(FILE_PATH)
elif FILE_PATH.endswith(".csv"):
    df = pd.read_csv(FILE_PATH)
else:
    print("Unsupported file type. Use .csv or .xlsx")

print("Shape (rows, columns):", df.shape)
print("\nColumn names:", list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())


PROGRAMME 1: Load and inspect the membership dataset
Shape (rows, columns): (1000, 8)

Column names: ['Checkin_ID', 'Member_ID', 'Membership_Tier', 'Gender', 'Checkin_Time', 'Workout_Duration_Mins', 'Fitness_Class', 'Monthly_Fee']

Data types:
Checkin_ID                object
Member_ID                 object
Membership_Tier           object
Gender                    object
Checkin_Time              object
Workout_Duration_Mins    float64
Fitness_Class             object
Monthly_Fee                int64
dtype: object

First 5 rows:
  Checkin_ID Member_ID Membership_Tier  Gender         Checkin_Time  \
0     G00000     M0316        Platinum   Other  2026-05-01 22:00:00   
1     G00001     M0234           Basic    Male  2025-05-01 07:00:00   
2     G00002     M0094            Gold  Female  2026-10-14 17:00:00   
3     G00003     M0204            Gold   Other  2025-10-09 15:00:00   
4     G00004     M0281           Basic    Male  2025-04-11 09:00:00   

   Workout_Duration_Mins Fitness_Cla

## PROGRAMME 2: Missing values and duplicate Checkin_ID entries

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 2: Missing values and duplicate Checkin_ID entries")
print("=" * 70)

print("Missing values per column:")
print(df.isnull().sum())

dup_count = df["Checkin_ID"].duplicated().sum()
print("\nDuplicate Checkin_ID entries:", dup_count)

if dup_count > 0:
    print("Rows that share a Checkin_ID (first 6 shown):")
    print(df[df["Checkin_ID"].duplicated(keep=False)].sort_values("Checkin_ID").head(6))
else:
    print("No duplicate Checkin_ID found.")



PROGRAMME 2: Missing values and duplicate Checkin_ID entries
Missing values per column:
Checkin_ID                 0
Member_ID                  0
Membership_Tier            0
Gender                     0
Checkin_Time               0
Workout_Duration_Mins     22
Fitness_Class            174
Monthly_Fee                0
dtype: int64

Duplicate Checkin_ID entries: 12
Rows that share a Checkin_ID (first 6 shown):
    Checkin_ID Member_ID Membership_Tier  Gender         Checkin_Time  \
39      G00039     M0170           Basic  Female  2026-08-14 14:00:00   
854     G00039     M0170           Basic  Female  2026-08-14 14:00:00   
66      G00066     M0372            Gold  Female  2025-01-31 05:00:00   
855     G00066     M0372            Gold  Female  2025-01-31 05:00:00   
110     G00110     M0174           Basic    Male  2026-05-23 14:00:00   
852     G00110     M0174           Basic    Male  2026-05-23 14:00:00   

     Workout_Duration_Mins Fitness_Class  Monthly_Fee  
39                

## PROGRAMME 3: Clean missing Workout_Duration_Mins values

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 3: Clean missing Workout_Duration_Mins values")
print("=" * 70)

# Remove duplicate check-ins first (keep the first copy)
data = df.drop_duplicates(subset="Checkin_ID", keep="first").copy()
print("Rows after removing duplicates:", len(data))

# Method chosen: fill with the MEDIAN.
# Reason: the median is not pulled up or down by very long or very short
# sessions, so it is a safe middle value for missing durations.
missing_duration = data["Workout_Duration_Mins"].isnull().sum()
median_duration = data["Workout_Duration_Mins"].median()

if missing_duration == 0:
    print("No missing workout durations, nothing to fill.")
elif missing_duration > 0:
    data["Workout_Duration_Mins"] = data["Workout_Duration_Mins"].fillna(median_duration)
    print("Missing durations filled:", missing_duration)
    print("Median used:", median_duration, "minutes")

# Missing Fitness_Class is labelled instead of guessed
class_missing = data["Fitness_Class"].isnull().sum()
data["Fitness_Class"] = data["Fitness_Class"].fillna("Not Specified")

print("\nMissing values after cleaning:")
print(data.isnull().sum())
print(data.head())

# Randomize: shuffle the row order (seed 42 so the shuffle can be repeated).
# Checkin_ID is kept, so sorting by it gives the original order back.
print("\nRandomizing the row order...")
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
print("Rows after shuffle:", len(data))
print(data.head())



PROGRAMME 3: Clean missing Workout_Duration_Mins values
Rows after removing duplicates: 988
Missing durations filled: 22
Median used: 72.0 minutes

Missing values after cleaning:
Checkin_ID               0
Member_ID                0
Membership_Tier          0
Gender                   0
Checkin_Time             0
Workout_Duration_Mins    0
Fitness_Class            0
Monthly_Fee              0
dtype: int64
  Checkin_ID Member_ID Membership_Tier  Gender         Checkin_Time  \
0     G00000     M0316        Platinum   Other  2026-05-01 22:00:00   
1     G00001     M0234           Basic    Male  2025-05-01 07:00:00   
2     G00002     M0094            Gold  Female  2026-10-14 17:00:00   
3     G00003     M0204            Gold   Other  2025-10-09 15:00:00   
4     G00004     M0281           Basic    Male  2025-04-11 09:00:00   

   Workout_Duration_Mins Fitness_Class  Monthly_Fee  
0                   30.0          HIIT          999  
1                   71.0          Spin         2499  
2 

## PROGRAMME 4: Convert Checkin_Time to datetime, extract weekday and hour

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 4: Convert Checkin_Time to datetime, extract weekday and hour")
print("=" * 70)

data["Checkin_Time"] = pd.to_datetime(data["Checkin_Time"])
data["Weekday"] = data["Checkin_Time"].dt.day_name()
data["Hour"] = data["Checkin_Time"].dt.hour
data["Month"] = data["Checkin_Time"].dt.to_period("M")

print(data[["Checkin_ID", "Checkin_Time", "Weekday", "Hour", "Month"]].head())



PROGRAMME 4: Convert Checkin_Time to datetime, extract weekday and hour
  Checkin_ID        Checkin_Time   Weekday  Hour    Month
0     G00213 2026-03-23 11:00:00    Monday    11  2026-03
1     G00331 2025-12-14 09:00:00    Sunday     9  2025-12
2     G00500 2025-05-24 06:00:00  Saturday     6  2025-05
3     G00309 2026-05-09 05:00:00  Saturday     5  2026-05
4     G00088 2026-02-22 15:00:00    Sunday    15  2026-02


## PROGRAMME 5: Total check-ins and average workout duration

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 5: Total check-ins and average workout duration")
print("=" * 70)

total_checkins = len(data)
avg_duration = data["Workout_Duration_Mins"].mean()

print("Total check-ins:", total_checkins)
print("Average workout duration:", round(avg_duration, 2), "minutes")



PROGRAMME 5: Total check-ins and average workout duration
Total check-ins: 988
Average workout duration: 72.47 minutes


## PROGRAMME 6: Visit frequency across membership tiers

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 6: Visit frequency across membership tiers")
print("=" * 70)

tier_order = ["Basic", "Gold", "Platinum"]
tier_visits = data.groupby("Membership_Tier").size().reindex(tier_order)
tier_members = data.groupby("Membership_Tier")["Member_ID"].nunique().reindex(tier_order)

tier_table = pd.DataFrame({
    "Total_Visits": tier_visits,
    "Unique_Members": tier_members,
})
tier_table["Visits_per_Member"] = (tier_table["Total_Visits"] / tier_table["Unique_Members"]).round(2)
tier_table["Share_of_Visits_%"] = (tier_table["Total_Visits"] / total_checkins * 100).round(1)

print(tier_table)
print("\nTier with the most visits:", tier_visits.idxmax())



PROGRAMME 6: Visit frequency across membership tiers
                 Total_Visits  Unique_Members  Visits_per_Member  \
Membership_Tier                                                    
Basic                     487             252               1.93   
Gold                      333             223               1.49   
Platinum                  168             138               1.22   

                 Share_of_Visits_%  
Membership_Tier                     
Basic                         49.3  
Gold                          33.7  
Platinum                      17.0  

Tier with the most visits: Basic


## PROGRAMME 7: Peak gym hours and busiest days of the week

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 7: Peak gym hours and busiest days of the week")
print("=" * 70)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hour_counts = data["Hour"].value_counts().sort_index()
day_counts = data["Weekday"].value_counts().reindex(day_order)

print("Check-ins by hour:")
print(hour_counts)
print("\nTop 3 peak hours:")
print(hour_counts.sort_values(ascending=False).head(3))
print("\nCheck-ins by weekday:")
print(day_counts)
print("\nBusiest day:", day_counts.idxmax())
print("Quietest day:", day_counts.idxmin())



PROGRAMME 7: Peak gym hours and busiest days of the week
Check-ins by hour:
Hour
5     60
6     57
7     72
8     51
9     42
10    54
11    50
12    51
13    69
14    52
15    41
16    46
17    45
18    59
19    40
20    44
21    57
22    48
23    50
Name: count, dtype: int64

Top 3 peak hours:
Hour
7     72
13    69
5     60
Name: count, dtype: int64

Check-ins by weekday:
Weekday
Monday       144
Tuesday      132
Wednesday    150
Thursday     148
Friday       120
Saturday     160
Sunday       134
Name: count, dtype: int64

Busiest day: Saturday
Quietest day: Friday


## PROGRAMME 8: Rank fitness classes by total attendance

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 8: Rank fitness classes by total attendance")
print("=" * 70)

# "Not Specified" is not a real class, so it is left out of the ranking
real_classes = data[data["Fitness_Class"] != "Not Specified"]
class_rank = real_classes["Fitness_Class"].value_counts()

print("Class ranking (most attended first):")
rank_number = 1
for class_name, attendance in class_rank.items():
    print(rank_number, "-", class_name, ":", attendance)
    rank_number = rank_number + 1



PROGRAMME 8: Rank fitness classes by total attendance
Class ranking (most attended first):
1 - Strength : 175
2 - Spin : 171
3 - Zumba : 161
4 - HIIT : 161
5 - Yoga : 148


## PROGRAMME 9: Average workout duration, male vs female

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 9: Average workout duration, male vs female")
print("=" * 70)

gender_avg = data.groupby("Gender")["Workout_Duration_Mins"].mean().round(2)
print(gender_avg)

male_avg = gender_avg["Male"]
female_avg = gender_avg["Female"]

if male_avg > female_avg:
    print("\nMale members work out longer by", round(male_avg - female_avg, 2), "minutes.")
elif female_avg > male_avg:
    print("\nFemale members work out longer by", round(female_avg - male_avg, 2), "minutes.")
else:
    print("\nMale and female members have the same average duration.")



PROGRAMME 9: Average workout duration, male vs female
Gender
Female    72.51
Male      74.25
Other     70.38
Name: Workout_Duration_Mins, dtype: float64

Male members work out longer by 1.74 minutes.


## PROGRAMME 10: Monthly membership revenue by tier

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 10: Monthly membership revenue by tier")
print("=" * 70)

# A member pays the fee once per month, not once per visit,
# so keep one row per member per month before adding the fees.
# Note: Monthly_Fee does not always match the tier in this dataset,
# so the fee written on each record is used as it is.
member_month = data.drop_duplicates(subset=["Member_ID", "Month"], keep="first")

monthly_revenue = member_month.groupby(["Month", "Membership_Tier"])["Monthly_Fee"].sum()
monthly_revenue = monthly_revenue.unstack(fill_value=0)
monthly_revenue = monthly_revenue[tier_order]
monthly_revenue["Total"] = monthly_revenue.sum(axis=1)

print(monthly_revenue.head(12))
print("\nTotal revenue by tier:")
print(member_month.groupby("Membership_Tier")["Monthly_Fee"].sum().reindex(tier_order))



PROGRAMME 10: Monthly membership revenue by tier
Membership_Tier  Basic   Gold  Platinum  Total
Month                                         
2025-01          23988  20988     12493  57469
2025-02          30980  24483       999  56462
2025-03          49472  16987     18490  84949
2025-04          31980  25984     14491  72455
2025-05          17989  24986      9495  52470
2025-06          32480  15989      9993  58462
2025-07          36979  23987      7494  68460
2025-08          40477  12991     10495  63963
2025-09          36977  25486      6994  69457
2025-10          25483  17488      8495  51466
2025-11          36477  22485      9995  68957
2025-12          41477  41477      8995  91949

Total revenue by tier:
Membership_Tier
Basic       763542
Gold        503691
Platinum    264345
Name: Monthly_Fee, dtype: int64


## PROGRAMME 11: Members with fewer than 2 visits per month

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 11: Members with fewer than 2 visits per month")
print("=" * 70)

# Average visits per month = total visits / number of months in the data
number_of_months = data["Month"].nunique()
member_visits = data["Member_ID"].value_counts()
visits_per_month = member_visits / number_of_months

low_engagement = visits_per_month[visits_per_month < 2]

print("Months in the data:", number_of_months)
print("Total members:", len(member_visits))
print("Low-engagement members (under 2 visits per month):", len(low_engagement))
print("Average check-ins per member:", round(member_visits.mean(), 2))

# The data is sparse, so every member is below 2 visits per month.
# A more useful group is members who checked in only once.
once_only = int((member_visits == 1).sum())
once_pct = once_only / len(member_visits) * 100
print("Members with only 1 check-in ever:", once_only, "(", round(once_pct, 1), "% )")

print("\nSample of low-engagement members:")
print(low_engagement.head())



PROGRAMME 11: Members with fewer than 2 visits per month
Months in the data: 22
Total members: 348
Low-engagement members (under 2 visits per month): 348
Average check-ins per member: 2.84
Members with only 1 check-in ever: 85 ( 24.4 % )

Sample of low-engagement members:
Member_ID
M0398    0.409091
M0081    0.409091
M0126    0.409091
M0236    0.363636
M0028    0.363636
Name: count, dtype: float64


## PROGRAMME 12: NumPy - median, variance and quartiles of workout duration

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 12: NumPy - median, variance and quartiles of workout duration")
print("=" * 70)

durations = data["Workout_Duration_Mins"].to_numpy(dtype=float)

median_value = np.median(durations)
variance_value = np.var(durations, ddof=1)
std_value = np.std(durations, ddof=1)
q1 = np.percentile(durations, 25)
q2 = np.percentile(durations, 50)
q3 = np.percentile(durations, 75)

print("Median            :", round(median_value, 2))
print("Variance          :", round(variance_value, 2))
print("Standard deviation:", round(std_value, 2))
print("Q1 (25%)          :", round(q1, 2))
print("Q2 (50%)          :", round(q2, 2))
print("Q3 (75%)          :", round(q3, 2))



PROGRAMME 12: NumPy - median, variance and quartiles of workout duration
Median            : 72.0
Variance          : 959.07
Standard deviation: 30.97
Q1 (25%)          : 47.0
Q2 (50%)          : 72.0
Q3 (75%)          : 98.0


## PROGRAMME 13: Unusually long or short workout sessions (mean and std method)

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 13: Unusually long or short workout sessions (mean and std method)")
print("=" * 70)

# A session is unusual if it is more than 2 standard deviations
# away from the average duration.
mean_value = np.mean(durations)
lower_limit = mean_value - 2 * std_value
upper_limit = mean_value + 2 * std_value

print("Mean duration:", round(mean_value, 2))
print("Lower limit  :", round(lower_limit, 2))
print("Upper limit  :", round(upper_limit, 2))


def session_type(minutes):
    if minutes < lower_limit:
        return "Unusually Short"
    elif minutes > upper_limit:
        return "Unusually Long"
    else:
        return "Normal"


data["Session_Type"] = data["Workout_Duration_Mins"].apply(session_type)
print("\nSession type counts:")
print(data["Session_Type"].value_counts())
print("Shortest session:", data["Workout_Duration_Mins"].min(), "minutes")
print("Longest session :", data["Workout_Duration_Mins"].max(), "minutes")

unusual = data[data["Session_Type"] != "Normal"]
if len(unusual) == 0:
    print("\nNo unusual sessions found. All sessions are within the limits.")
else:
    print("\nUnusual sessions:")
    print(unusual.head())



PROGRAMME 13: Unusually long or short workout sessions (mean and std method)
Mean duration: 72.47
Lower limit  : 10.53
Upper limit  : 134.41

Session type counts:
Session_Type
Normal    988
Name: count, dtype: int64
Shortest session: 20.0 minutes
Longest session : 129.0 minutes

No unusual sessions found. All sessions are within the limits.


## PROGRAMME 14: Workout-intensity label (Short, Moderate, Extended)

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 14: Workout-intensity label (Short, Moderate, Extended)")
print("=" * 70)

# Thresholds used:
#   Short    : less than 45 minutes
#   Moderate : 45 to 90 minutes
#   Extended : more than 90 minutes
def intensity_label(minutes):
    if minutes < 45:
        return "Short"
    elif minutes <= 90:
        return "Moderate"
    else:
        return "Extended"


data["Workout_Intensity"] = data["Workout_Duration_Mins"].apply(intensity_label)

print(data[["Checkin_ID", "Workout_Duration_Mins", "Workout_Intensity"]].head())
intensity_counts = data["Workout_Intensity"].value_counts().reindex(["Short", "Moderate", "Extended"])
print("\nIntensity counts:")
print(intensity_counts)
print("\nIntensity share (%):")
print((intensity_counts / len(data) * 100).round(1))



PROGRAMME 14: Workout-intensity label (Short, Moderate, Extended)
  Checkin_ID  Workout_Duration_Mins Workout_Intensity
0     G00213                  126.0          Extended
1     G00331                   27.0             Short
2     G00500                   80.0          Moderate
3     G00309                   48.0          Moderate
4     G00088                   35.0             Short

Intensity counts:
Workout_Intensity
Short       223
Moderate    455
Extended    310
Name: count, dtype: int64

Intensity share (%):
Workout_Intensity
Short       22.6
Moderate    46.1
Extended    31.4
Name: count, dtype: float64


## PROGRAMME 15: Five facility management and retention recommendations

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 15: Five facility management and retention recommendations")
print("=" * 70)

top_hours = hour_counts.sort_values(ascending=False).head(3).index.tolist()
quiet_hours = hour_counts.sort_values().head(3).index.tolist()
busy_day = day_counts.idxmax()
quiet_day = day_counts.idxmin()
top_class = class_rank.index[0]
bottom_class = class_rank.index[-1]
unspecified_pct = (data["Fitness_Class"] == "Not Specified").sum() / len(data) * 100

if once_pct >= 25:
    retention_note = "A large share of members never came back after one visit, so welcome offers are urgent."
elif once_pct >= 10:
    retention_note = "A fair share of members visited only once, so follow-up offers will help."
else:
    retention_note = "Few members visited only once, so focus on keeping regular members active."

if unspecified_pct >= 10:
    class_note = "Many check-ins have no class recorded, so make class selection compulsory at check-in."
else:
    class_note = "Most check-ins record a class, so keep the current process."

recommendations = [
    "1. Staffing: add staff near peak hours " + str(sorted(top_hours)) + " and on " + busy_day
    + ". Keep cover lighter around quiet hours " + str(sorted(quiet_hours)) + ".",
    "2. Classes: " + top_class + " is the most attended, so add more sessions. "
    + bottom_class + " is the least attended, so review its timing or trainer.",
    "3. Off-peak offers: promote discounts on " + quiet_day + " and in quiet hours to spread the crowd.",
    "4. Retention: " + str(once_only) + " members (" + str(round(once_pct)) + "%) visited only once. " + retention_note,
    "5. Data quality: " + str(round(unspecified_pct)) + "% of check-ins have no class. " + class_note
    + " Also match Monthly_Fee with Membership_Tier before using tier revenue for pricing.",
]

for line in recommendations:
    print(line)



PROGRAMME 15: Five facility management and retention recommendations
1. Staffing: add staff near peak hours [5, 7, 13] and on Saturday. Keep cover lighter around quiet hours [9, 15, 19].
2. Classes: Strength is the most attended, so add more sessions. Yoga is the least attended, so review its timing or trainer.
3. Off-peak offers: promote discounts on Friday and in quiet hours to spread the crowd.
4. Retention: 85 members (24%) visited only once. A fair share of members visited only once, so follow-up offers will help.
5. Data quality: 17% of check-ins have no class. Many check-ins have no class recorded, so make class selection compulsory at check-in. Also match Monthly_Fee with Membership_Tier before using tier revenue for pricing.


## PROGRAMME 16 (EXTRA): Time of day of check-ins (Morning, Afternoon, Evening, Night)

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 16 (EXTRA): Time of day of check-ins (Morning, Afternoon, Evening, Night)")
print("=" * 70)

def time_of_day(hour):
    if hour < 12:
        return "Morning"
    elif hour < 17:
        return "Afternoon"
    elif hour < 21:
        return "Evening"
    else:
        return "Night"


data["Time_of_Day"] = data["Hour"].apply(time_of_day)

print(data[["Checkin_Time", "Hour", "Time_of_Day"]].head())
time_counts = data["Time_of_Day"].value_counts().reindex(["Morning", "Afternoon", "Evening", "Night"])
print("\nCheck-ins by time of day:")
print(time_counts)
print("\nMost popular time of day:", time_counts.idxmax())



PROGRAMME 16 (EXTRA): Time of day of check-ins (Morning, Afternoon, Evening, Night)
         Checkin_Time  Hour Time_of_Day
0 2026-03-23 11:00:00    11     Morning
1 2025-12-14 09:00:00     9     Morning
2 2025-05-24 06:00:00     6     Morning
3 2026-05-09 05:00:00     5     Morning
4 2026-02-22 15:00:00    15   Afternoon

Check-ins by time of day:
Time_of_Day
Morning      386
Afternoon    259
Evening      188
Night        155
Name: count, dtype: int64

Most popular time of day: Morning


## PROGRAMME 17 (EXTRA): Weekday vs weekend check-ins

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 17 (EXTRA): Weekday vs weekend check-ins")
print("=" * 70)

def day_type(day_name):
    if day_name == "Saturday":
        return "Weekend"
    elif day_name == "Sunday":
        return "Weekend"
    else:
        return "Weekday"


data["Day_Type"] = data["Weekday"].apply(day_type)

print(data[["Checkin_Time", "Weekday", "Day_Type"]].head())
type_counts = data["Day_Type"].value_counts()
print("\nCheck-ins by day type:")
print(type_counts)

# Weekdays are 5 days and weekends are 2 days, so compare the daily average
weekday_per_day = type_counts["Weekday"] / 5
weekend_per_day = type_counts["Weekend"] / 2
print("\nAverage check-ins per weekday  :", round(weekday_per_day, 1))
print("Average check-ins per weekend day:", round(weekend_per_day, 1))

if weekend_per_day > weekday_per_day:
    print("Weekend days are busier than weekdays.")
elif weekend_per_day < weekday_per_day:
    print("Weekdays are busier than weekend days.")
else:
    print("Weekdays and weekend days are equally busy.")



PROGRAMME 17 (EXTRA): Weekday vs weekend check-ins
         Checkin_Time   Weekday Day_Type
0 2026-03-23 11:00:00    Monday  Weekday
1 2025-12-14 09:00:00    Sunday  Weekend
2 2025-05-24 06:00:00  Saturday  Weekend
3 2026-05-09 05:00:00  Saturday  Weekend
4 2026-02-22 15:00:00    Sunday  Weekend

Check-ins by day type:
Day_Type
Weekday    694
Weekend    294
Name: count, dtype: int64

Average check-ins per weekday  : 138.8
Average check-ins per weekend day: 147.0
Weekend days are busier than weekdays.


## PROGRAMME 18: Save the cleaned + randomized data and the results workbook

In [ ]:
print("\n" + "=" * 70)
print("PROGRAMME 18: Save the cleaned + randomized data and the results workbook")
print("=" * 70)

# Cleaned and randomized dataset (original columns plus the new ones)
data.to_csv("12_gym_membership_clean_shuffled.csv", index=False)
print("Saved: 12_gym_membership_clean_shuffled.csv")

cleaning_log = pd.DataFrame({
    "Step": ["Rows in original file", "Duplicate Checkin_ID rows removed", "Rows after cleaning",
             "Missing Workout_Duration_Mins filled with median", "Missing Fitness_Class set to Not Specified",
             "Row order"],
    "Result": [len(df), dup_count, len(data), missing_duration, class_missing, "Shuffled (seed 42)"],
})
cleaning_log["Result"] = cleaning_log["Result"].astype(str)

hour_table = hour_counts.rename("Check_ins").reset_index()
day_table = day_counts.rename("Check_ins").reset_index()
class_table = class_rank.rename("Total_Attendance").reset_index()
class_table.insert(0, "Rank", range(1, len(class_table) + 1))
revenue_table = monthly_revenue.reset_index()
revenue_table["Month"] = revenue_table["Month"].astype(str)
low_table = low_engagement.rename("Avg_Visits_per_Month").round(3).reset_index()
low_table.columns = ["Member_ID", "Avg_Visits_per_Month"]
low_table["Total_Visits"] = low_table["Member_ID"].map(member_visits)
stats_table = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Variance", "Std dev", "Q1", "Q2", "Q3", "Lower limit (mean-2sd)", "Upper limit (mean+2sd)"],
    "Value": [mean_value, median_value, variance_value, std_value, q1, q2, q3, lower_limit, upper_limit],
}).round(2)
gender_table = gender_avg.rename("Avg_Duration_Mins").reset_index()
intensity_table = intensity_counts.rename("Sessions").reset_index()
intensity_table.columns = ["Workout_Intensity", "Sessions"]
time_table = time_counts.rename("Check_ins").reset_index()
time_table.columns = ["Time_of_Day", "Check_ins"]
rec_table = pd.DataFrame({"Recommendation": recommendations})

with pd.ExcelWriter("Project12_Results.xlsx") as writer:
    cleaning_log.to_excel(writer, sheet_name="Cleaning_Log", index=False)
    hour_table.to_excel(writer, sheet_name="Peak_Hours", index=False)
    day_table.to_excel(writer, sheet_name="Busiest_Days", index=False)
    class_table.to_excel(writer, sheet_name="Class_Popularity", index=False)
    tier_table.reset_index().to_excel(writer, sheet_name="Tier_Comparison", index=False)
    gender_table.to_excel(writer, sheet_name="Gender_Duration", index=False)
    revenue_table.to_excel(writer, sheet_name="Monthly_Revenue", index=False)
    low_table.to_excel(writer, sheet_name="Low_Engagement", index=False)
    stats_table.to_excel(writer, sheet_name="Duration_Stats", index=False)
    intensity_table.to_excel(writer, sheet_name="Workout_Intensity", index=False)
    time_table.to_excel(writer, sheet_name="Time_of_Day", index=False)
    rec_table.to_excel(writer, sheet_name="Recommendations", index=False)
print("Saved: Project12_Results.xlsx")

# In Google Colab, download the saved files to your computer
try:
    from google.colab import files
    files.download("Project12_Results.xlsx")
    files.download("12_gym_membership_clean_shuffled.csv")
except ImportError:
    print("Not in Colab: files are saved in the current folder.")
